# qlib benchmark run — free GPU (Kaggle / Colab)

Runs `scripts/gpu_run.py` against the private `Riyu-D1/quant-research` repo.

**Setup once:** store a GitHub PAT (read+write on the repo) as secret `GH_TOKEN`
- Kaggle: *Add-ons → Secrets*
- Colab: *key icon → Secrets*

Enable a GPU accelerator before running: Kaggle *Settings → Accelerator (T4/P100)*, Colab *Runtime → Change runtime type → T4*.

In [ ]:
import os
ON_KAGGLE = os.path.exists('/kaggle')
ON_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
WORK = '/kaggle/working' if ON_KAGGLE else ('/content' if ON_COLAB else os.path.expanduser('~/cloud-run'))
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print('platform:', 'kaggle' if ON_KAGGLE else 'colab' if ON_COLAB else 'local', '| workdir:', WORK)

In [ ]:
def get_secret(name):
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if ON_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        return os.environ.get(name)
    return os.environ.get(name)

GH_TOKEN = get_secret('GH_TOKEN')
if not GH_TOKEN:
    raise RuntimeError('Set GH_TOKEN secret (PAT with repo access)')
REPO_URL = f'https://{GH_TOKEN}@github.com/Riyu-D1/quant-research.git'
if not os.path.exists('quant-research'):
    !git clone $REPO_URL
# idempotent: re-running cells after a kernel restart must not fail on %cd
if os.path.basename(os.getcwd()) != 'quant-research':
    %cd quant-research

In [ ]:
!pip -q install uv
!uv python install 3.11
!uv sync --frozen --no-dev
import sys, subprocess
print(subprocess.run(['.venv/bin/python', '-c',
    'import torch; print(torch.__version__, "cuda:", torch.cuda.is_available())'],
    capture_output=True, text=True).stdout)

In [ ]:
# edit this list; see `python scripts/gpu_run.py --list`
MODELS = 'transformer_cn hist_cn'
!.venv/bin/python scripts/gpu_run.py --models $MODELS

In [ ]:
!cat results/LEADERBOARD.md 2>/dev/null; ls -la results/runs/

In [ ]:
# optional: push results back to a dated branch (set PUSH_RESULTS=1)
import os, datetime
if os.environ.get('PUSH_RESULTS') == '1' and GH_TOKEN:
    br = f"results/{'kaggle' if ON_KAGGLE else 'colab'}-{datetime.date.today()}"
    !git checkout -b $br
    !git add results/ EXPERIMENTS.md   # mlruns/ + data/ are gitignored on purpose
    !git -c user.name='gpu-run' -c user.email='noreply@users.noreply.github.com' commit -m 'gpu run results' || true
    !git push origin $br
    print('pushed branch', br)
else:
    print('PUSH_RESULTS not set — results stay in the session; download results/runs/*.json manually')